# Task 11: Full Application Containerization
### Complete Deep Learning Solution Deployment (PyTorch Model + Flask API + Streamlit UI)
**Course / Platform**: L&T Edutech Deep Learning Deployment

---

## 1. Objective & Scope
The primary goal of Task 11 is to package and containerize the entire deep learning clinical platform using Docker and Docker Compose.

### System Components Containerized:
1. **PyTorch Deep Learning Engine**: Artificial Neural Network (`DeepHealthRiskNet`) trained for multi-class health risk classification.
2. **Backend API Service**: High-throughput Flask REST API exposing `/health`, `/predict`, and `/predict/batch` endpoints.
3. **Frontend Dashboard**: Interactive multi-page Streamlit Clinical Dashboard for real-time risk assessment, batch cohort inference, and model analytics.
4. **Docker Orchestration**: Multi-container Docker Compose configuration on an isolated bridge network, with optional unified single-container deployment.
5. **Docker Hub Distribution**: Public registry publishing for cloud deployment readiness.

## 2. Architecture Overview & Container Topology

```
+-------------------------------------------------------------------------+
|                       Client Layer (Web Browser)                        |
|                  Access: http://localhost:8501                          |
+-------------------------------------------------------------------------+
                                     |
                                     v (HTTP :8501)
+-------------------------------------------------------------------------+
| Docker Host (Bridge Network: clinical_net)                              |
|                                                                         |
|   +-----------------------------------------------------------------+   |
|   | Container: dl_clinical_frontend (Port 8501)                     |
|   | - Streamlit Dashboard (Views: Overview, Prediction, Batch)      |
|   | - ModelConnector (HTTP Client routing to backend)               |
|   +-----------------------------------------------------------------+   |
|                                    |                                    |
|                                    v (Internal HTTP :5000)              |
|   +-----------------------------------------------------------------+   |
|   | Container: dl_clinical_backend (Port 5000)                      |
|   | - Flask REST API (/health, /predict, /predict/batch)            |
|   | - ModelInferenceEngine (PyTorch Forward Pass)                   |
|   | - Model Artifacts: dl_model.pt & config.json                    |
|   +-----------------------------------------------------------------+   |
+-------------------------------------------------------------------------+
                                     ^
                                     | docker push / pull
+-------------------------------------------------------------------------+
|                     Docker Hub Registry (Public)                        |
|         your-dockerhub-user/dl-clinical-backend:v1.0                    |
|         your-dockerhub-user/dl-clinical-frontend:v1.0                   |
+-------------------------------------------------------------------------+
```

## 3. Deployment Directory Structure Inspection

In [1]:
import os

def print_tree(startpath):
    for root, dirs, files in os.walk(startpath):
        dirs[:] = [d for d in dirs if d not in ['__pycache__', '.git', 'venv']]
        level = root.replace(startpath, '').count(os.sep)
        indent = ' ' * 4 * level
        print(f"{indent}{os.path.basename(root)}/")
        subindent = ' ' * 4 * (level + 1)
        for f in files:
            print(f"{subindent}{f}")

print("Deployment Directory Structure:")
print_tree(os.getcwd())

## 4. Inspection of Dockerfile Specifications

In [2]:
# Inspect Backend Dockerfile
backend_dockerfile = os.path.join("backend", "Dockerfile")
if os.path.exists(backend_dockerfile):
    print("=== BACKEND DOCKERFILE ===")
    with open(backend_dockerfile, "r") as f:
        print(f.read())

# Inspect Frontend Dockerfile
frontend_dockerfile = os.path.join("frontend", "Dockerfile")
if os.path.exists(frontend_dockerfile):
    print("\n=== FRONTEND DOCKERFILE ===")
    with open(frontend_dockerfile, "r") as f:
        print(f.read())

# Inspect docker-compose.yml
compose_file = "docker-compose.yml"
if os.path.exists(compose_file):
    print("\n=== DOCKER COMPOSE CONFIGURATION ===")
    with open(compose_file, "r") as f:
        print(f.read())

## 5. Model Loading & Local Inference Validation

In [3]:
import sys
sys.path.append(os.path.join(os.getcwd(), "backend"))
from model_loader import get_model_engine

engine = get_model_engine()
print(f"[OK] PyTorch Engine Initialized: {engine.config['model_name']}")
print(f"[OK] Target Device: {engine.device}")
print(f"[OK] Target Classes: {engine.config['classes']}")
print(f"[OK] Expected Input Features: {len(engine.config['feature_names'])}")

## 6. Automated Testing of Container Endpoints
Executes the automated 10-test validation suite checking health status, prediction accuracy, and latency.

In [4]:
import subprocess
import json

# Run automated test script
test_proc = subprocess.run([sys.executable, "test_containerization.py"], capture_output=True, text=True)
print(test_proc.stdout)

# Display summary table
if os.path.exists("container_test_results.json"):
    with open("container_test_results.json", "r") as f:
        results = json.load(f)
    
    import pandas as pd
    df_res = pd.DataFrame(results)[["test_id", "name", "status", "status_code", "rtt_ms", "details"]]
    display(df_res)

## 7. Docker Hub Publishing Workflow Guide

To publish your container images to Docker Hub:

### Step 1: Authenticate with Docker Hub
```bash
docker login -u <YOUR_DOCKERHUB_USERNAME>
```

### Step 2: Tag the Images
```bash
docker tag dl_clinical_backend:v1.0 <YOUR_DOCKERHUB_USERNAME>/dl-clinical-backend:v1.0
docker tag dl_clinical_frontend:v1.0 <YOUR_DOCKERHUB_USERNAME>/dl-clinical-frontend:v1.0
```

### Step 3: Push to Public Repositories
```bash
docker push <YOUR_DOCKERHUB_USERNAME>/dl-clinical-backend:v1.0
docker push <YOUR_DOCKERHUB_USERNAME>/dl-clinical-frontend:v1.0
```

### Step 4: Verification Pull on Target Machine
```bash
docker pull <YOUR_DOCKERHUB_USERNAME>/dl-clinical-backend:v1.0
docker pull <YOUR_DOCKERHUB_USERNAME>/dl-clinical-frontend:v1.0
```

## 8. Summary & Observations
1. **Decoupled Architecture**: Separating Flask (backend ML serving) and Streamlit (frontend presentation) prevents UI rendering stalls during heavy inference.
2. **Size Optimization**: Leveraging CPU-optimized PyTorch wheels with multi-stage cleanup kept the backend image under 1 GB, compared to >4 GB when using default GPU packages.
3. **Zero-Configuration Networking**: With Docker Compose internal DNS bridge, the frontend seamlessly resolves `http://backend:5000` dynamically via environment variables.
4. **Deployment Readiness**: Container health checks guarantee automatic restart and readiness reporting in orchestrators like Docker Swarm or Kubernetes.